# Feature Engineering

This notebook learns multi-word phrases from the preprocessed speeches, filters them, and builds phrase-enhanced document-term matrices (TF-IDF and raw counts) for downstream analysis (distinctive terms, LDA).

Named entity recognition now lives in `04_named_entity_recognition.ipynb`.

**Inputs:** `ungdc_preprocessed.parquet` (from notebook 02) — `sentences` column (normalized, stopwords preserved).

**Outputs** (`data/processed/`): `ungdc_features.parquet`, `ungdc_tfidf_matrix.npz`, `ungdc_count_matrix.npz`, fitted vectorizers, and phrase models.

## Set up

In [1]:
from pathlib import Path
import sys
from collections import Counter

import pandas as pd
import numpy as np
from IPython.display import display

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import save_npz

# Gensim phrase detection
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS

import joblib

# Project paths
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Shared preprocessing configuration (stopwords, phrase filter)
sys.path.insert(0, str(PROJECT_DIR / "src"))
import text_processing as tp

pd.set_option("display.max_rows", 100)

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)

Project directory: /Users/johncreighton/CS6220/International Political Discourse Mining
Processed data directory: /Users/johncreighton/CS6220/International Political Discourse Mining/data/processed


In [2]:
# Load preprocessed corpus
input_path = PROCESSED_DIR / "ungdc_preprocessed.parquet"

metadata = pd.read_parquet(
    input_path,
    columns=["filename", "country_code", "session", "year", "sentences", "tokens"]
)

# Parquet returns nested arrays; convert to plain lists of strings
metadata["sentences"] = metadata["sentences"].apply(
    lambda doc: [list(sentence) for sentence in doc]
)

print(f"Loaded {len(metadata):,} speeches")
print(f"Countries/entities: {metadata['country_code'].nunique():,}")
print(f"Year range: {metadata['year'].min()}–{metadata['year'].max()}")
print(f"Sentences: {metadata['sentences'].apply(len).sum():,}")
print(f"Stopwords (from src/text_processing.py): {len(tp.STOPWORDS):,}")

display(metadata.head())

Loaded 11,141 speeches
Countries/entities: 200
Year range: 1946–2025
Sentences: 1,258,550
Stopwords (from src/text_processing.py): 325


,filename,country_code,session,year,sentences,tokens
0,ARG_01_1946.txt,ARG,1,1946,"[[at, the, resumption, of, the, first, session...","[resumption, general, assembly, argentine, wis..."
1,AUS_01_1946.txt,AUS,1,1946,"[[the, general, assembly, of, the, united, nat...","[general, assembly, united, nations, meeting, ..."
2,BEL_01_1946.txt,BEL,1,1946,"[[the, principal, organs, of, the, united, nat...","[principal, organs, united, nations, functioni..."
3,BLR_01_1946.txt,BLR,1,1946,"[[as, more, than, a, year, has, elapsed, since...","[year, elapsed, united, nations, charter, sign..."
4,BOL_01_1946.txt,BOL,1,1946,"[[coming, to, this, platform, where, so, many,...","[coming, platform, distinguished, eloquent, sp..."


## Detect n-grams

### Phrase model parameters

| Parameter | Value | Meaning |
|---|---|---|
| Training unit | sentence | Phrases cannot span sentence boundaries |
| `scoring` | `npmi` | Normalized pointwise mutual information, bounded in [-1, 1] |
| `threshold` | 0.5 | Keep word pairs with NPMI ≥ 0.5 (strong association) |
| `min_count` | 5 | Ignore word pairs seen fewer than 5 times |
| `connector_words` | gensim's English connectors + all stopwords | Connectors may appear inside a phrase (*peace and security*, *rule of law*) but never at its start or end |
| Trigram stage | none | See rationale |
| Phrase filter | safety check, same edge rule | See *Filter phrases* |

**Why NPMI instead of gensim's default scorer.** The default score, `(count(ab) - min_count) / (count(a) * count(b)) * vocab_size`, penalizes very frequent words and depends on vocabulary size. With it, *united states* (because *states* is extremely common) and *peace and security* fell below the threshold depending on minor preprocessing changes. NPMI measures how much more often two words co-occur than chance, normalized to [-1, 1], so the threshold has the same meaning regardless of corpus size and is easier to justify.

**Why no trigram stage.** We compared configurations on the full corpus by measuring, for 20 reference concepts, the share of occurrences that end up as the concept's own token (*capture rate*; see the diagnostic below). A second phrase pass always lowered capture, because it absorbs core concepts into longer, rarer tokens (*respect for human rights*, *protection of human rights*, *human rights council*). Bigram-only NPMI at 0.5 gave the highest capture (e.g. human rights 0.62 → 1.00, nuclear weapons 0.58 → 0.97, arms race 0.64 → 0.98 relative to the previous bigram+trigram setup). Phrases containing a connector (*rule of law*, *heads of state*) are still learned in one pass.

**Trade-offs.**
- Compositional pairs of very common words (*economic development*) are not merged; both words still appear as separate features.
- Long proper names (*weapons of mass destruction*, *people's republic of china*) are split; these are better handled by entity normalization (notebook 04) than by n-gram features.

In [3]:
# Each sentence is a separate sequence of tokens
sentences = [
    sentence
    for document in metadata["sentences"]
    for sentence in document
]

# Train the phrase model
bigram_model = Phrases(
    sentences,
    min_count=5,
    threshold=0.5,
    scoring="npmi",
    connector_words=tp.phrase_connector_words(),
    delimiter="__"
)

# Freeze to a smaller, faster model for applying phrases
bigram_phraser = bigram_model.freeze()

print(f"Phrase model trained on {len(sentences):,} sentences.")
print(f"Learned phrases: {len(bigram_phraser.phrasegrams):,}")

Phrase model trained on 1,258,550 sentences.
Learned phrases: 9,849


## Filter phrases

Statistical association also picks up grammatical combinations that are not concepts (*has been*, *should be*, *my country*, *would like*, *not only*). In the previous version of this notebook these became features in TF-IDF and the count matrix.

Because every stopword is passed to gensim as a *connector word*, the phrase model cannot learn a phrase that starts or ends with a stopword in the first place. This matters beyond removing junk: phrase detection is greedy, so a grammatical phrase such as *session of the general* would otherwise consume the *general* of *general assembly*.

`tp.apply_phrases` re-checks the same rule as a safety net. To show what is being excluded, the next cell trains a model with gensim's default connectors only and lists the most frequent phrases it would have produced that violate the rule.

Stopwords are then removed from the remaining single tokens.

In [4]:
def create_analysis_tokens(document_sentences):
    """Apply phrase models and the phrase filter, then remove stopwords."""
    tokens = []
    for sentence in document_sentences:
        tokens.extend(
            tp.apply_phrases(sentence, bigram_phraser)
        )
    return tp.remove_stopwords(tokens)


metadata["analysis_tokens"] = metadata["sentences"].apply(create_analysis_tokens)
metadata["analysis_text"] = metadata["analysis_tokens"].apply(" ".join)

display(
    metadata[["filename", "tokens", "analysis_tokens"]].head(5)
)

,filename,tokens,analysis_tokens
0,ARG_01_1946.txt,"[resumption, general, assembly, argentine, wis...","[resumption, general__assembly, argentine, wis..."
1,AUS_01_1946.txt,"[general, assembly, united, nations, meeting, ...","[general__assembly, united__nations, meeting, ..."
2,BEL_01_1946.txt,"[principal, organs, united, nations, functioni...","[principal__organs, united__nations, functioni..."
3,BLR_01_1946.txt,"[year, elapsed, united, nations, charter, sign...","[year, elapsed, united__nations, charter, sign..."
4,BOL_01_1946.txt,"[coming, platform, distinguished, eloquent, sp...","[coming, platform, distinguished, eloquent, sp..."


In [5]:
# For comparison: phrases learned when only gensim's default connector words are used
unconstrained_phraser = Phrases(
    sentences,
    min_count=5,
    threshold=0.5,
    scoring="npmi",
    connector_words=ENGLISH_CONNECTOR_WORDS,
    delimiter="__"
).freeze()

unconstrained_counts = Counter()
for sentence in sentences:
    for token in unconstrained_phraser[sentence]:
        if "__" in token:
            unconstrained_counts[token] += 1

excluded = pd.DataFrame(
    [
        {"phrase": phrase.replace("__", " "), "frequency": count}
        for phrase, count in unconstrained_counts.most_common()
        if not tp.is_valid_phrase(phrase)
    ][:30]
)

print(f"Phrase types with default connectors: {len(unconstrained_counts):,}")
print(f"...of which start or end with a stopword: {sum(not tp.is_valid_phrase(p) for p in unconstrained_counts):,}")
print("\nMost frequent excluded phrases:")
display(excluded)

del unconstrained_phraser, unconstrained_counts

Phrase types with default connectors: 9,900
...of which start or end with a stopword: 808

Most frequent excluded phrases:


,phrase,frequency
0,it is,85162
1,has been,34836
2,have been,27343
3,must be,23674
4,should be,22596
5,my country,16665
6,as well,16599
7,not only,13545
8,more than,13530
9,my delegation,12626


## Inspect resulting phrases

In [6]:
phrase_counts = Counter()
phrase_documents = Counter()

for tokens in metadata["analysis_tokens"]:
    phrases = [token for token in tokens if "__" in token]

    phrase_counts.update(phrases)
    phrase_documents.update(set(phrases))


top_phrases = pd.DataFrame(
    [
        {
            "phrase": phrase.replace("__", " "),
            "frequency": count,
            "document_frequency": phrase_documents[phrase]
        }
        for phrase, count in phrase_counts.most_common(75)
    ]
)

print(f"Phrase types kept: {len(phrase_counts):,}")
display(top_phrases)

Phrase types kept: 9,496


,phrase,frequency,document_frequency
0,united nations,153233,11050
1,general assembly,40931,10366
2,international community,32727,8432
3,security council,29045,8522
4,human rights,26204,7116
5,secretary general,24874,9031
6,united states,23351,5566
7,developing countries,22118,5984
8,peace and security,18190,7279
9,middle east,15544,6255


### Concept capture diagnostic

Phrase detection is greedy and context-dependent, so a concept can be split across several tokens or not merged at all. For a set of reference concepts, the table compares how often the word sequence appears in the text with how often it ends up as its own token. A capture rate near 1 means the concept is represented consistently in the document-term matrix.

Hyphenated compounds such as *selfdetermination* and *nonproliferation* are single tokens after notebook 02, so they are not listed here. *economic development* is expected to be 0: it is a compositional pair of very frequent words and is intentionally left as two features.

This is a property of the *features*, not of the corpus. Trend analysis of specific concepts should count the concept directly in the normalized text rather than rely on phrase tokens.

In [7]:
reference_concepts = [
    "united nations", "general assembly", "security council",
    "international community", "human rights", "developing countries",
    "peace and security", "nuclear weapons", "cold war",
    "climate change", "sustainable development", "international law",
    "middle east", "south africa", "soviet union", "third world",
    "arms race", "rule of law", "use of force", "economic development",
]

in_text = Counter()
for sentence in sentences:
    padded = " " + " ".join(sentence) + " "
    for concept in reference_concepts:
        in_text[concept] += padded.count(" " + concept + " ")

capture = pd.DataFrame(
    [
        {
            "concept": concept,
            "occurrences_in_text": in_text[concept],
            "as_own_token": phrase_counts[concept.replace(" ", "__")],
            "inside_longer_phrase": sum(
                count for phrase, count in phrase_counts.items()
                if concept.replace(" ", "__") in phrase
                and phrase != concept.replace(" ", "__")
            ),
        }
        for concept in reference_concepts
    ]
)
capture["capture_rate"] = (
    capture["as_own_token"] / capture["occurrences_in_text"]
).round(2)

display(capture)

,concept,occurrences_in_text,as_own_token,inside_longer_phrase,capture_rate
0,united nations,153230,153233,0,1.00
1,general assembly,40931,40931,0,1.00
2,security council,29156,29045,0,1.00
3,international community,32727,32727,0,1.00
4,human rights,26200,26204,0,1.00
5,developing countries,22199,22118,0,1.00
6,peace and security,18334,18190,0,0.99
7,nuclear weapons,9546,9226,187,0.97
8,cold war,4446,3231,0,0.73
9,climate change,10828,10567,0,0.98


## TF-IDF

In [8]:
# Create TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.80
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    metadata["analysis_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of TF-IDF features:", len(tfidf_vectorizer.vocabulary_))

feature_names = tfidf_vectorizer.get_feature_names_out()
n_phrase_features = sum("__" in f for f in feature_names)
print(f"Phrase features: {n_phrase_features:,} ({n_phrase_features / len(feature_names):.1%})")
# Terms dropped by max_df: present in more than 80% of speeches
doc_freq = Counter(
    token for tokens in metadata["analysis_tokens"] for token in set(tokens)
)
too_common = sorted(
    (term for term, df in doc_freq.items() if df > 0.80 * len(metadata)),
    key=lambda term: -doc_freq[term]
)
print(f"Terms dropped by max_df=0.80 ({len(too_common)}):", too_common)

TF-IDF matrix shape: (11141, 34957)
Number of TF-IDF features: 34957
Phrase features: 8,528 (24.4%)


Terms dropped by max_df=0.80 (34): ['world', 'united__nations', 'not', 'international', 'countries', 'peace', 'people', 'country', 'development', 'general__assembly', 'new', 'support', 'efforts', 'economic', 'time', 'organization', 'political', 'no', 'states', 'work', 'cooperation', 'government', 'continue', 'like', 'year', 'order', 'state', 'today', 'secretary__general', 'make', 'president', 'security', 'years', 'need']


## CountVectorizer

In [9]:
# Create count-based document-term matrix (useful for LDA)
count_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.80
)

count_matrix = count_vectorizer.fit_transform(
    metadata["analysis_text"]
)

print("Count matrix shape:", count_matrix.shape)
print("Number of count features:", len(count_vectorizer.vocabulary_))

Count matrix shape: (11141, 34957)
Number of count features: 34957


## Save Outputs

In [10]:
# Save document-level metadata and transformed text
feature_columns = [
    "filename",
    "country_code",
    "session",
    "year",
    "analysis_text",
    "analysis_tokens"
]

feature_data = metadata[feature_columns].copy()

feature_data.to_parquet(
    PROCESSED_DIR / "ungdc_features.parquet",
    index=False
)

# Save sparse document-term matrices
save_npz(
    PROCESSED_DIR / "ungdc_tfidf_matrix.npz",
    tfidf_matrix
)

save_npz(
    PROCESSED_DIR / "ungdc_count_matrix.npz",
    count_matrix
)

# Save fitted vectorizers so we can recover feature names later
joblib.dump(
    tfidf_vectorizer,
    PROCESSED_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(
    count_vectorizer,
    PROCESSED_DIR / "count_vectorizer.joblib"
)

# Save phrase model for reuse
joblib.dump(
    bigram_phraser,
    PROCESSED_DIR / "bigram_phraser.joblib"
)

print("Saved feature data and models to:", PROCESSED_DIR)

Saved feature data and models to: /Users/johncreighton/CS6220/International Political Discourse Mining/data/processed
